In [ ]:
# =============================================================================
# AIR QUALITY DATA IMPORT
# Reads daily air quality data from Excel, reshapes it to long format,
# cleans/renames variables, and exports to an ArcGIS geodatabase table.
# =============================================================================

import pandas as pd
import arcpy
import os

# -----------------------------------------------------------------------------
# CONFIGURATION
# Update these paths and settings as needed
# -----------------------------------------------------------------------------

# Excel files to import (add more paths to the list if needed)
EXCEL_FILES = [
    r"F:\GIS\PROJECTS\Monitoring\AirQuality\SurveyData\2025_AQ_Data.xlsx"
]

SHEET_NAME = 'daily'  # Sheet name to read from each Excel file

# Output geodatabase and table
GDB_PATH    = r"F:\GIS\PROJECTS\Monitoring\AirQuality\AirQuality_Map\AirQuality_Map.gdb"
OUTPUT_TABLE = "AQ_temp"
OUTPUT_PATH  = os.path.join(GDB_PATH, OUTPUT_TABLE)

# Columns to drop — weather/metadata variables not needed for air quality analysis
COLUMNS_TO_DROP = ['RH', 'BP', 'RWD', 'RWD.1', 'RWS', 'Tmp']

# Rename raw variable codes to descriptive pollutant names
VARIABLE_RENAME_MAP = {
    'COmax'      : 'CO - 1 hr max (ppm)',
    'max8hrCO'   : 'CO - 8 hr max (ppm)',
    'NO2_avg'    : 'NO2 - annual mean (ppm)',
    'NO2max'     : 'NO2 - 1 hr max (ppm)',
    'O3max'      : 'O3 - 1 hr max (ppm)',
    'max8hrO3'   : 'O3 - 8 hr max (ppm)',
    'PM10avg'    : 'PM10 - 24 hr avg (mg/m3)',
    'PM2.5avg'   : 'PM2.5 - 24 hr avg (mg/m3)',
    'PM2.5avg.1' : 'PM2.5 - 24 hr avg (mg/m3)',
}

# Map pandas dtypes to ArcGIS field types
# Uncomment 'int64' or 'datetime64[ns]' if those column types appear in your data
DTYPE_TO_ARCGIS = {
    # 'int64'          : 'LONG',
    'float64'        : 'DOUBLE',
    'object'         : 'TEXT',
    'string'         : 'TEXT',
    # 'datetime64[ns]' : 'DATE',
}


# -----------------------------------------------------------------------------
# STEP 1: READ EXCEL FILES
# Reads each file's 'daily' sheet with a two-row header (site name + variable)
# -----------------------------------------------------------------------------

print("Reading Excel files...")
dataframes = []

for file_path in EXCEL_FILES:
    df = pd.read_excel(file_path, sheet_name=SHEET_NAME, header=[0, 1])
    dataframes.append(df)
    print(f"  Loaded: {file_path}")

# Combine all files into one DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)
print(f"  Total rows loaded: {len(combined_df)}")


# -----------------------------------------------------------------------------
# STEP 2: FLATTEN MULTI-LEVEL COLUMN HEADERS
# Excel had two header rows, creating a MultiIndex like ('SITE', 'COmax').
# Flatten to plain strings like 'SITE_COmax' so standard pandas operations work.
# -----------------------------------------------------------------------------

print("\nFlattening multi-level column headers...")

combined_df.columns = [
    f"{site}_{variable}" if variable else site
    for site, variable in combined_df.columns
]


# -----------------------------------------------------------------------------
# STEP 3: RESHAPE FROM WIDE TO LONG FORMAT
# Each row currently has one date and many pollutant columns.
# After melting, each row will represent one date + one pollutant measurement.
#
# Before melt:  date | SITE_COmax | SITE_NO2max | ...
# After melt:   date | site       | variable    | value
# -----------------------------------------------------------------------------

print("Reshaping data from wide to long format...")

long_df = combined_df.melt(
    id_vars=['SITE_date'],   # Column to keep as the row identifier
    var_name='site_variable', # Temporary column holding 'SITE_COmax' style names
    value_name='value'
)

# Split 'SITE_COmax' into separate 'id' and 'variable' columns
long_df[['id', 'variable']] = long_df['site_variable'].str.split('_', n=1, expand=True)

# Drop the now-redundant combined column and rename the date column
long_df = long_df.drop(columns=['site_variable'])
long_df = long_df.rename(columns={'SITE_date': 'date'})

# Reorder columns for readability
long_df = long_df[['date', 'id', 'variable', 'value']]


# -----------------------------------------------------------------------------
# STEP 4: CLEAN AND FILTER DATA
# - Remove weather/metadata rows not needed for air quality analysis
# - Rename variable codes to descriptive pollutant names
# - Drop rows with no measurement value
# - Ensure numeric values are stored as float
# -----------------------------------------------------------------------------

print("Cleaning and filtering data...")

# Remove rows for weather variables (RH, BP, wind, temp, etc.)
long_df = long_df[~long_df['variable'].isin(COLUMNS_TO_DROP)]

# Apply descriptive pollutant names
long_df['variable'] = long_df['variable'].replace(VARIABLE_RENAME_MAP)

# Format date as YYYY-MM-DD string (required for ArcGIS text field)
long_df['date'] = pd.to_datetime(long_df['date']).dt.strftime('%Y-%m-%d')

# Drop rows with no measurement, then convert value to numeric
long_df = long_df.dropna(subset=['value'])
long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce').astype('float64')

print(f"  Rows after cleaning: {len(long_df)}")


# -----------------------------------------------------------------------------
# STEP 5: EXPORT TO ARCGIS GEODATABASE TABLE
# - Delete the existing table if it already exists
# - Create a fresh table with fields matching the DataFrame columns
# - Insert all rows using an InsertCursor
# -----------------------------------------------------------------------------

print(f"\nExporting to geodatabase table: {OUTPUT_PATH}")

# Delete existing table to avoid conflicts
if arcpy.Exists(OUTPUT_PATH):
    arcpy.management.Delete(OUTPUT_PATH)
    print(f"  Deleted existing table: {OUTPUT_TABLE}")

# Create a new empty table in the geodatabase
arcpy.management.CreateTable(GDB_PATH, OUTPUT_TABLE)

# Add one field per DataFrame column, using the dtype-to-ArcGIS mapping
print("  Adding fields...")
for col_name, dtype in long_df.dtypes.items():
    arcgis_type = DTYPE_TO_ARCGIS.get(str(dtype), 'TEXT')  # Default to TEXT if dtype is unrecognized

    if arcgis_type == 'TEXT':
        arcpy.management.AddField(OUTPUT_PATH, col_name, arcgis_type, field_length=255)
    else:
        arcpy.management.AddField(OUTPUT_PATH, col_name, arcgis_type)

# Insert all rows from the DataFrame into the geodatabase table
print("  Inserting rows...")
with arcpy.da.InsertCursor(OUTPUT_PATH, long_df.columns.tolist()) as cursor:
    for _, row in long_df.iterrows():
        cursor.insertRow(row.tolist())

print(f"\nDone. Table '{OUTPUT_TABLE}' created and populated in:\n  {GDB_PATH}")

Reading Excel files...
  Loaded: F:\GIS\PROJECTS\Monitoring\AirQuality\SurveyData\2025_AQ_Data.xlsx
  Total rows loaded: 365

Flattening multi-level column headers...
Reshaping data from wide to long format...
Cleaning and filtering data...
  Rows after cleaning: 4369

Exporting to geodatabase table: F:\GIS\PROJECTS\Monitoring\AirQuality\AirQuality_Map\AirQuality_Map.gdb\AQ_temp
  Adding fields...
  Inserting rows...

Done. Table 'AQ_temp' created and populated in:
  F:\GIS\PROJECTS\Monitoring\AirQuality\AirQuality_Map\AirQuality_Map.gdb
